# Install Kubectl and K3d

In [ ]:
%%bash

sudo pacman -S docker kubectl
yay -S k3d

sudo pacman -S helm

# Create k3d cluster

In [ ]:
%%bash

PARENT_DIR="$(dirname "$PWD")"

k3d cluster create dev \
  --volume "$PARENT_DIR/k3d/registries.yaml:/etc/rancher/k3s/registries.yaml@all" \
  --volume "$PARENT_DIR/k3d/certs/cla321-local-ca.crt:/etc/rancher/k3s/certs/cla321-local-ca.crt@all"

# Install Headlamp for K3d

In [ ]:
%%bash

sudo pacman -S helm

helm repo add headlamp https://kubernetes-sigs.github.io/headlamp/
helm repo update

helm install headlamp headlamp/headlamp \
  --namespace headlamp \
  --create-namespace

kubectl get pods -n headlamp
kubectl get svc -n headlamp

kubectl apply -f k8s/headlamp/ingress.yaml

# Create K3d root account

In [ ]:
import socket
import subprocess
from pathlib import Path

NAMESPACE = "headlamp"
SERVICE_ACCOUNT = "root"
K8S_TOKEN_PATH = Path("assets/k8s_token.txt")


def ensure_headlamp_root_account() -> None:
    subprocess.run(
        [
            "kubectl",
            "create",
            "serviceaccount",
            SERVICE_ACCOUNT,
            "-n",
            NAMESPACE,
        ],
        check=True,
    )

    subprocess.run(
        [
            "kubectl",
            "create",
            "clusterrolebinding",
            "headlamp-root-cluster-admin",
            "--clusterrole=cluster-admin",
            f"--serviceaccount={NAMESPACE}:{SERVICE_ACCOUNT}",
        ],
        check=True,
    )

def create_headlamp_token() -> str:
    result = subprocess.run(
        [
            "kubectl",
            "create",
            "token",
            SERVICE_ACCOUNT,
            "-n",
            NAMESPACE,
            "--duration=24h",
        ],
        check=True,
        capture_output=True,
        text=True,
    )

    return result.stdout.strip()

In [ ]:
ensure_headlamp_root_account()

In [ ]:
k8s_token = create_headlamp_token()
K8S_TOKEN_PATH.parent.mkdir(parents=True, exist_ok=True)
K8S_TOKEN_PATH.write_text(f"{k8s_token}\n", encoding="utf-8")
print(f'k8s token: {k8s_token}')